# 🧪 TCGA-LUAD: Uji 5 Skenario (Lanjutan Ekstraksi Fitur) — **Revisi sesuai review**

Notebook ini **melanjutkan** `TCGA_LUAD_Feature_Extraction`. Seluruh data berasal dari file fitur `.h5` hasil ekstraksi:

```
MyDrive/TCGA-LUAD-WSI/
├── Fitur_Swin_Tiny/{WSI_ID}.h5      (768-dim)    ← input
├── Fitur_DenseNet201/{WSI_ID}.h5    (1920-dim)   ← input
└── Hasil_Skenario/                  ← OUTPUT (tabel CSV/Excel, gambar PNG, cache run)
```

| No | Skenario | Isi |
|----|----------|-----|
| 1 | Model Utama | Swin-T+ABMIL vs Swin-T+CLAM → AUROC, AUPRC, Sensitivitas, Spesifisitas, dst. |
| 2 | Perbandingan dengan CNN | Swin-T (pilihan CV) vs DenseNet201 |
| 3 | Sensitivitas Hyperparameter | Learning rate, Optimizer, Batch size (satu faktor diubah per kali) |
| 4 | Efisiensi Komputasi | Parameter, FLOPs/patch, waktu inferensi MIL, puncak memori GPU |
| 5 | Signifikansi Statistik | DeLong, McNemar, Bootstrap ≥1000× (95% CI), α = 0,05 |
| 6 | Analisis kualitatif (3.6.2) | Distribusi atensi + error analysis FP/FN |

### Protokol evaluasi (menjawab review)
1. **5-fold cross-validation tingkat pasien** pada data latih (train+validation = 160 WSI). Fold distratifikasi pada **label pasien**, sehingga komposisi Mutant per fold hampir sempurna.
2. **Seleksi** (model Swin-T terbaik, konfigurasi hyperparameter, seed) memakai **AUROC out-of-fold (OOF)**, bukan data uji.
3. **Threshold** (Youden's J) dihitung dari **prediksi OOF** gabungan seluruh fold, lalu dipakai tetap pada data uji.
4. **Model akhir** dilatih ulang pada **seluruh 160 WSI** dengan jumlah epoch = median epoch terbaik antar fold, lalu diuji **satu kali** pada data uji (test tidak dipakai untuk keputusan apa pun).
5. Early stopping di dalam tiap fold memakai **subset dalam (inner)** dari data latih fold itu, sehingga prediksi fold held-out tetap bersih (tanpa kebocoran seleksi).
6. SD yang dilaporkan = **SD sampel (ddof = 1)** antar seed.
7. Metrik untuk data tidak seimbang: ditambah **AUPRC**, **akurasi seimbang**, dan **F1**.
8. Skenario 2 adalah **perbandingan**; kode otomatis memperingatkan jika urutan model di OOF-CV dan di uji berbeda, agar dibahas terbuka.

### Keputusan desain lain
- **Label positif** = `Mutant` (1), `Wildtype` = 0. Level analisis = **WSI**.
- **ABMIL** = gated attention MIL (Ilse et al.). **CLAM** = CLAM-SB (Lu et al.) dengan instance-level clustering loss (CE, top-k = 8, bobot bag 0,7 / instance 0,3).
- **DenseNet201** = fitur DenseNet201 + agregator MIL yang **sama** dengan Swin-T pilihan CV.
- **Batch size** = jumlah slide (bag) per satu update optimizer (**gradient accumulation**), bukan batch tensor.
- Hasil training di-**cache** (`runs_cache.pkl`) → aman bila runtime terputus.

## 📦 Sel 0: Dependencies, Drive, dan Path

In [ ]:
!pip install h5py tqdm scikit-learn scipy seaborn timm openpyxl -q
print("✅ Dependencies installed")

In [ ]:
import os
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("ℹ️  Bukan Colab — lewati mount Drive (isi ROOT_MANUAL di sel berikutnya)")

if IN_COLAB:
    if os.path.isdir('/content/drive/MyDrive'):
        print("✅ Drive sudah ter-mount")
    else:
        for attempt, force in enumerate([False, True], 1):
            try:
                drive.mount('/content/drive', force_remount=force); break
            except Exception as e:
                print(f"⚠️ Percobaan {attempt} gagal: {type(e).__name__}: {e}")
        print("✅ Drive mounted" if os.path.isdir('/content/drive/MyDrive') else "❌ Mount gagal — coba Runtime → Disconnect and delete runtime")

In [ ]:
import os
ROOT_MANUAL = None   # contoh: "/content/drive/MyDrive/TCGA-LUAD-WSI"

def find_root():
    if ROOT_MANUAL:
        return ROOT_MANUAL
    base = "/content/drive/MyDrive"
    for c in [f"{base}/TCGA-LUAD-WSI", f"{base}/TCGA-LUAD"]:
        if os.path.isdir(f"{c}/Fitur_Swin_Tiny"):
            return c
    if os.path.isdir(base):
        for dp, dn, _ in os.walk(base):
            if "Fitur_Swin_Tiny" in dn:
                return dp
            if dp[len(base):].count(os.sep) >= 3:
                dn[:] = []
    return f"{base}/TCGA-LUAD-WSI"

ROOT       = find_root()
PATCH_DIR  = f"{ROOT}/Patches_Macenko"          # hanya dipakai sebagai cadangan pencarian label
FEAT_SWIN  = f"{ROOT}/Fitur_Swin_Tiny"
FEAT_DENSE = f"{ROOT}/Fitur_DenseNet201"
RESULT_DIR = f"{ROOT}/Hasil_Skenario"
FIG_DIR    = f"{RESULT_DIR}/figures"
CKPT_DIR   = f"{RESULT_DIR}/checkpoints"

for d in (FEAT_SWIN, FEAT_DENSE):
    if not os.path.isdir(d):
        raise FileNotFoundError(f"Folder fitur tidak ditemukan: {d}\nJalankan notebook ekstraksi fitur dulu / isi ROOT_MANUAL.")
for d in (RESULT_DIR, FIG_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

print("ROOT        :", ROOT)
print("Swin  .h5   :", len([f for f in os.listdir(FEAT_SWIN)  if f.endswith('.h5')]))
print("Dense .h5   :", len([f for f in os.listdir(FEAT_DENSE) if f.endswith('.h5')]))
print("Output      :", RESULT_DIR)

## ⚙️ Sel 1: Konfigurasi Eksperimen & Tema Visual Cerah

In [ ]:
# ====================== KONFIGURASI ======================
SEED            = 42
N_SEEDS         = 3                       # jumlah seed untuk Skenario 1 & 2 (rerata ± SD sampel)
SEEDS           = [SEED + i for i in range(N_SEEDS)]
N_FOLDS         = 5                       # cross-validation tingkat pasien pada data latih (train+val)
ES_FRACTION     = 0.20                    # porsi inner-validation (untuk early stopping) di dalam tiap fold
EPOCHS          = 60
PATIENCE        = 12
MAX_PATCH_TRAIN = 4000                    # subsampel patch/slide saat training; evaluasi memakai SEMUA patch
POSITIVE_CLASS  = 'Mutant'

DEFAULT_HP = dict(lr=1e-4, optimizer='AdamW', batch_size=4, weight_decay=1e-4)
HP_GRID = {                               # Skenario 3: satu faktor diubah, faktor lain = DEFAULT_HP
    'lr'        : [1e-5, 5e-5, 1e-4, 5e-4, 1e-3],
    'optimizer' : ['Adam', 'AdamW', 'SGD', 'RMSprop'],
    'batch_size': [1, 4, 8, 16],
}

CLAM_K, CLAM_BAG_W = 8, 0.7               # CLAM-SB: top-k instance & bobot bag loss (instance = 1 - bobot bag)
THRESH_MODE = 'youden_oof'                # 'youden_oof' (Youden dari prediksi OOF) atau angka tetap, mis. 0.5
N_BOOT      = 2000                        # bootstrap (≥1000)
ALPHA       = 0.05
# =========================================================

In [ ]:
import os, io, gc, time, random, pickle, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, h5py
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, average_precision_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy.stats import norm, chi2, binomtest
from tqdm.auto import tqdm
from IPython.display import display
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else "CPU (lambat — disarankan GPU)")

# ---------- Tema cerah ----------
C_ABMIL, C_CLAM, C_DENSE = '#FF7B54', '#2D9CDB', '#22B573'
C_MUT, C_WT = '#FF4D6D', '#00B4D8'
BG, GRID = '#FBFDFF', '#E8EEF5'
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': 'white', 'savefig.facecolor': BG,
    'axes.edgecolor': '#CFD8E3', 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.titleweight': 'bold', 'axes.titlesize': 12, 'axes.labelsize': 10, 'font.size': 10,
    'legend.frameon': True, 'legend.facecolor': 'white', 'legend.edgecolor': '#E0E6ED',
    'figure.dpi': 110,
})

def color_of(agg, bb):
    return C_DENSE if bb == 'Dense' else (C_ABMIL if agg == 'ABMIL' else C_CLAM)
def mname(agg, bb):
    return f"{'Swin-T' if bb == 'Swin' else 'DenseNet201'}+{agg}"
def light_cmap(color):
    return LinearSegmentedColormap.from_list('c', ['#FFFFFF', color])
def savefig(fig, name):
    p = f"{FIG_DIR}/{name}.png"
    fig.savefig(p, dpi=200, bbox_inches='tight'); print("💾", p)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

## 🗂️ Sel 2: Scan Patches_Macenko → wsi_df, Metadata, dan Fold CV Tingkat Pasien

In [ ]:
# ============================================================
# SCAN PATCHES_MACENKO → BUAT wsi_df   (kode dari notebook ekstraksi fitur)
# Mendukung 2 format:
#   (a) ZIP per WSI : {split}/{class}/{WSI_ID}.zip   ← format dataset Anda
#   (b) Folder      : {split}/{class}/{WSI_ID}/*.png
# Folder lain (qc, manifests, reference, dll.) diabaikan.
# ============================================================
import io, zipfile
from PIL import Image

SPLIT_ALIASES = {'train':'train','training':'train',
                 'val':'validation','valid':'validation','validation':'validation',
                 'test':'test','testing':'test'}
CLASS_ALIASES = {'mutant':'Mutant','mut':'Mutant','egfr_mutant':'Mutant','egfr-mutant':'Mutant',
                 'wildtype':'Wildtype','wild_type':'Wildtype','wild-type':'Wildtype',
                 'wt':'Wildtype','egfr_wildtype':'Wildtype','egfr-wildtype':'Wildtype'}
IMG_EXT = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

def _norm(s): return s.strip().lower().replace(' ', '_')

def is_img_name(n):
    base = os.path.basename(n)
    return (os.path.splitext(base)[1].lower() in IMG_EXT
            and not base.startswith('.') and '__MACOSX' not in n)

def list_zip_images(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        return [n for n in zf.namelist() if is_img_name(n)]

def read_img_from_zip(zip_path, name):
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(name) as f:
            return Image.open(io.BytesIO(f.read())).convert('RGB')

META_CSV = f"{RESULT_DIR}/wsi_df_cache.csv"     # cache hasil scan → run berikutnya instan (hapus file ini untuk scan ulang)
if os.path.exists(META_CSV):
    wsi_df = pd.read_csv(META_CSV); print(f"♻️ wsi_df dari cache: {len(wsi_df)} WSI")
else:
    entries, skipped, ignored_dirs = [], [], []
    for sp_name in sorted(os.listdir(PATCH_DIR)):
        sp_dir = os.path.join(PATCH_DIR, sp_name)
        split = SPLIT_ALIASES.get(_norm(sp_name))
        if not os.path.isdir(sp_dir):
            continue
        if split is None:
            ignored_dirs.append(sp_name); continue
        for c_name in sorted(os.listdir(sp_dir)):
            c_dir = os.path.join(sp_dir, c_name)
            cls = CLASS_ALIASES.get(_norm(c_name))
            if cls is None or not os.path.isdir(c_dir):
                continue
            for e in sorted(os.listdir(c_dir)):
                ep = os.path.join(c_dir, e)
                if e.lower().endswith('.zip'):
                    entries.append((split, cls, e[:-4], ep, 'zip'))
                elif os.path.isdir(ep):
                    entries.append((split, cls, e, ep, 'dir'))

    print(f"🔎 Ditemukan {len(entries)} kandidat WSI (zip/folder). Membaca daftar patch...")
    if ignored_dirs:
        print(f"   (folder non-split diabaikan: {ignored_dirs})")

    wsi_info = []
    for split, cls, wsi_id, src, kind in tqdm(entries, desc="Scan"):
        try:
            if kind == 'zip':
                names = list_zip_images(src)
                sample = names[0] if names else None
            else:
                names = [str(q) for q in Path(src).rglob('*') if q.suffix.lower() in IMG_EXT]
                sample = names[0] if names else None
            if not names:
                skipped.append((wsi_id, 'tidak ada gambar')); continue
            if kind == 'zip':
                im = read_img_from_zip(src, sample)
            else:
                im = Image.open(sample)
            w, h, mode = im.width, im.height, im.mode
        except Exception as ex:
            skipped.append((wsi_id, f'{type(ex).__name__}: {ex}')); continue
        wsi_info.append({
            'wsi_id': wsi_id, 'patient_id': wsi_id[:12], 'split': split, 'label': cls,
            'n_patches': len(names), 'patch_w': w, 'patch_h': h, 'img_mode': mode,
            'wsi_dir': src,            # path ZIP atau folder (nama kolom dipertahankan)
            'src_type': kind, 'sample_patch': sample,
        })

    if not wsi_info:
        print("❌ Tidak ada WSI valid. Isi PATCH_DIR:", sorted(os.listdir(PATCH_DIR)))
        print("   Struktur diharapkan: Patches_Macenko/{train|test}/{Mutant|Wildtype}/{WSI_ID}.zip")
        raise RuntimeError("wsi_df kosong — cek struktur folder / PATCH_DIR di atas.")

    wsi_df = pd.DataFrame(wsi_info)
    if skipped:
        print(f"\n⚠️  {len(skipped)} entri dilewati, contoh: {skipped[:3]}")
    wsi_df.to_csv(META_CSV, index=False)

print(f"\n📊 Total WSI (hasil scan): {len(wsi_df)} | Total patch: {wsi_df['n_patches'].sum():,}")
print(wsi_df.groupby(['split', 'label']).agg(WSI=('wsi_id', 'count'), Patch=('n_patches', 'sum')))

# Cek kebocoran data: pasien yang sama berada di >1 split
leak = wsi_df.groupby('patient_id')['split'].nunique()
leak = leak[leak > 1]
if len(leak):
    print(f"\n⚠️  {len(leak)} pasien punya slide di >1 split (kebocoran data train/test):")
    display(wsi_df[wsi_df['patient_id'].isin(leak.index)][['patient_id', 'wsi_id', 'split', 'label']])
else:
    print("\n✅ Tidak ada kebocoran pasien antar split.")
dup = wsi_df[wsi_df['wsi_id'].duplicated(keep=False)]
if len(dup):
    print("⚠️  WSI_ID dobel:"); display(dup[['wsi_id', 'split', 'label']])

# ---- hanya WSI yang punya KEDUA file fitur (.h5) yang dipakai ----
ids_s = {e.name[:-3] for e in os.scandir(FEAT_SWIN)  if e.name.endswith('.h5')}
ids_d = {e.name[:-3] for e in os.scandir(FEAT_DENSE) if e.name.endswith('.h5')}
has_feat = wsi_df.wsi_id.isin(ids_s & ids_d)
if (~has_feat).any():
    print(f"⚠️ {(~has_feat).sum()} WSI hasil scan tidak punya H5 Swin+DenseNet201 → dikeluarkan: {wsi_df.wsi_id[~has_feat].tolist()[:3]}")
extra = (ids_s | ids_d) - set(wsi_df.wsi_id)
if extra: print(f"ℹ️ {len(extra)} file H5 tanpa entri di Patches_Macenko (diabaikan): {sorted(extra)[:3]}")
df = wsi_df[has_feat].reset_index(drop=True).copy()

assert len(df) > 0, "Tidak ada WSI valid yang punya file fitur."
df['y'] = (df['label'] == POSITIVE_CLASS).astype(int)
df['subset'] = np.where(df['split'].isin(['train', 'validation']), 'train', 'test')   # 'train' = pool latih (train+val) untuk CV

print(f"\n📊 Total WSI: {len(df)} | Pasien: {df.patient_id.nunique()} | Total patch: {df.n_patches.sum():,}")
display(df.groupby(['subset', 'label']).agg(WSI=('wsi_id', 'count'), Pasien=('patient_id', 'nunique'), Patch=('n_patches', 'sum')))
ov = set(df[df.subset == 'train'].patient_id) & set(df[df.subset == 'test'].patient_id)
print(f"{'✅' if not ov else '⚠️'} Irisan pasien latih ∩ uji: {len(ov)}")

ids_pool = df[df.subset == 'train'].wsi_id.tolist()      # 160 WSI (latih + validasi) → CV
ids_test = df[df.subset == 'test'].wsi_id.tolist()
Y      = dict(zip(df.wsi_id, df.y))
PAT_OF = dict(zip(df.wsi_id, df.patient_id))
PAT_Y  = df.groupby('patient_id').y.max().to_dict()      # label tingkat pasien (Mutant bila ada slide Mutant)
y_pool = np.array([Y[w] for w in ids_pool]); y_test = np.array([Y[w] for w in ids_test])
df_test = df[df.subset == 'test'].set_index('wsi_id').loc[ids_test].reset_index()
assert len(set(y_test)) == 2 and len(set(y_pool)) == 2

def patient_folds(ids, n_splits, seed):
    # K-fold terstratifikasi pada LABEL PASIEN → tidak ada pasien yang terbagi antar fold
    pats = np.array(sorted({PAT_OF[w] for w in ids})); lab = np.array([PAT_Y[p] for p in pats]); out = []
    for tr, ho in StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(pats, lab):
        st, sh = set(pats[tr]), set(pats[ho])
        out.append(([w for w in ids if PAT_OF[w] in st], [w for w in ids if PAT_OF[w] in sh]))
    return out

def patient_split(ids, frac, seed):
    pats = sorted({PAT_OF[w] for w in ids}); lab = [PAT_Y[p] for p in pats]
    a, b = train_test_split(pats, test_size=frac, stratify=lab, random_state=seed); sa, sb = set(a), set(b)
    return [w for w in ids if PAT_OF[w] in sa], [w for w in ids if PAT_OF[w] in sb]

FOLDS = patient_folds(ids_pool, N_FOLDS, SEED)
fold_tab = pd.DataFrame([dict(Fold=k + 1, WSI_latih=len(tr), Mutant_latih=sum(Y[w] for w in tr), WSI_heldout=len(ho), Mutant_heldout=sum(Y[w] for w in ho),
                              Wildtype_heldout=len(ho) - sum(Y[w] for w in ho)) for k, (tr, ho) in enumerate(FOLDS)])
print(f"\n🧩 {N_FOLDS}-fold CV tingkat pasien pada {len(ids_pool)} WSI (Mutant total: {int(y_pool.sum())}; target per fold ≈ {y_pool.sum() / N_FOLDS:.1f})")
display(fold_tab)
for k, (tr, ho) in enumerate(FOLDS):
    assert not ({PAT_OF[w] for w in tr} & {PAT_OF[w] for w in ho}), "Kebocoran pasien antar fold!"
    assert len({Y[w] for w in ho}) == 2, "Fold held-out harus berisi kedua kelas."
print("✅ Tidak ada pasien yang sama pada fold latih dan held-out")

In [ ]:
# ---------- Muat semua fitur ke RAM (float16 agar hemat memori) ----------
est = {}
for bb, d in [('Swin', FEAT_SWIN), ('Dense', FEAT_DENSE)]:
    with h5py.File(f"{d}/{df.wsi_id.iloc[0]}.h5", 'r') as hf:
        est[bb] = hf['features'].shape[1]
total_gb = sum(df.n_patches.sum() * dim * 2 for dim in est.values()) / 1e9
print(f"Perkiraan memori fitur (float16): {total_gb:.2f} GB")
try:
    import psutil
    avail = psutil.virtual_memory().available / 1e9
    print(f"RAM tersedia: {avail:.1f} GB")
    if total_gb > 0.85 * avail:
        print("⚠️ Memori mungkin tidak cukup. Gunakan runtime High-RAM atau kurangi jumlah patch per slide saat ekstraksi.")
except ImportError:
    pass

BANK = {'Swin': {}, 'Dense': {}}
for bb, d in [('Swin', FEAT_SWIN), ('Dense', FEAT_DENSE)]:
    for wid in tqdm(df.wsi_id, desc=f"Muat fitur {bb}"):
        with h5py.File(f"{d}/{wid}.h5", 'r') as hf:
            BANK[bb][wid] = torch.from_numpy(hf['features'][:].astype(np.float16))
FEAT_DIM = est
print("Dimensi fitur:", FEAT_DIM)

# n_patches dari scan ZIP bisa berbeda dari jumlah fitur di H5 → pakai jumlah fitur sebenarnya
df['n_patches'] = df.wsi_id.map(lambda w: BANK['Swin'][w].shape[0])
mism = [w for w in df.wsi_id if BANK['Swin'][w].shape[0] != BANK['Dense'][w].shape[0]]
print(f"{'⚠️' if mism else '✅'} Jumlah patch Swin ≠ DenseNet201 pada {len(mism)} WSI" + (f": {mism[:3]}" if mism else ""))
df_test = df[df.subset == 'test'].set_index('wsi_id').loc[ids_test].reset_index()
print(f"Patch/slide (H5): median {int(df.n_patches.median())} | min {int(df.n_patches.min())} | maks {int(df.n_patches.max())}")

## 🧠 Sel 3: Model MIL (ABMIL & CLAM-SB), Metrik, dan Prosedur 5-Fold CV + Latih Ulang

In [ ]:
METRICS = ['AUROC', 'AUPRC', 'Sensitivitas', 'Spesifisitas', 'Akurasi', 'Akurasi_Seimbang', 'F1']

class GatedAttention(nn.Module):
    def __init__(self, L=512, D=256, dropout=0.25):
        super().__init__()
        self.a = nn.Sequential(nn.Linear(L, D), nn.Tanh(),    nn.Dropout(dropout))
        self.b = nn.Sequential(nn.Linear(L, D), nn.Sigmoid(), nn.Dropout(dropout))
        self.c = nn.Linear(D, 1)
    def forward(self, x):                       # (N, L) -> (N, 1)
        return self.c(self.a(x) * self.b(x))

class ABMIL(nn.Module):
    # Attention-based deep MIL dengan gated attention (Ilse et al., 2018)
    def __init__(self, in_dim, hid=512, att=256, n_classes=2, dropout=0.25):
        super().__init__()
        self.fc  = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Dropout(dropout))
        self.att = GatedAttention(hid, att, dropout)
        self.cls = nn.Linear(hid, n_classes)
    def forward(self, h, label=None, instance_eval=False):
        h = self.fc(h)
        A = F.softmax(self.att(h).t(), dim=1)   # (1, N)
        M = A @ h
        return self.cls(M), {'attention': A, 'inst_loss': torch.zeros((), device=h.device)}

class CLAM_SB(nn.Module):
    # CLAM single-attention-branch (Lu et al., 2021), binary tanpa subtyping
    def __init__(self, in_dim, hid=512, att=256, n_classes=2, dropout=0.25, k_sample=CLAM_K):
        super().__init__()
        self.fc  = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Dropout(dropout))
        self.att = GatedAttention(hid, att, dropout)
        self.bag_cls  = nn.Linear(hid, n_classes)
        self.inst_cls = nn.ModuleList([nn.Linear(hid, 2) for _ in range(n_classes)])
        self.k = k_sample
    def _inst_loss(self, A, h, clf):
        k = min(self.k, A.shape[1])
        top_p = h[torch.topk(A,  k, dim=1)[1][-1]]
        top_n = h[torch.topk(-A, k, dim=1)[1][-1]]
        tg = torch.cat([torch.ones(k), torch.zeros(k)]).long().to(h.device)
        return F.cross_entropy(clf(torch.cat([top_p, top_n])), tg)
    def forward(self, h, label=None, instance_eval=False):
        h = self.fc(h)
        A = F.softmax(self.att(h).t(), dim=1)
        inst = torch.zeros((), device=h.device)
        if instance_eval and label is not None:
            inst = self._inst_loss(A, h, self.inst_cls[int(label)])
        M = A @ h
        return self.bag_cls(M), {'attention': A, 'inst_loss': inst}

def build_model(agg, in_dim):
    return ABMIL(in_dim) if agg == 'ABMIL' else CLAM_SB(in_dim)

def make_opt(name, params, lr, wd):
    if name == 'Adam':    return torch.optim.Adam(params, lr=lr, weight_decay=wd)
    if name == 'AdamW':   return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    if name == 'SGD':     return torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    if name == 'RMSprop': return torch.optim.RMSprop(params, lr=lr, weight_decay=wd)
    raise ValueError(name)

@torch.no_grad()
def predict(model, ids, bb):
    model.eval(); out = []
    for wid in ids:
        x = BANK[bb][wid].to(DEVICE).float()
        logits, _ = model(x)
        out.append(F.softmax(logits, 1)[0, 1].item())      # P(Mutant)
    return np.array(out)

def logloss(y, p, eps=1e-7):
    p = np.clip(p, eps, 1 - eps); return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def youden_thr(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[int(np.argmax(tpr - fpr))])

def cls_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    return dict(AUROC=roc_auc_score(y, p), AUPRC=average_precision_score(y, p), Sensitivitas=sens, Spesifisitas=spec,
                Akurasi=(tp + tn) / len(y), Akurasi_Seimbang=float(np.nanmean([sens, spec])), F1=f1_score(y, pred, zero_division=0))

def sd(v):                                    # SD sampel (ddof = 1)
    v = np.asarray(v, float); return float(np.std(v, ddof=1)) if len(v) > 1 else 0.0

def train_one(agg, bb, hp, seed, tr_ids, es_ids=None, fixed_epochs=None):
    # es_ids ada  → early stopping pada AUROC es_ids (inner-validation), bobot terbaik dipakai
    # es_ids None → latih tepat `fixed_epochs` epoch (untuk model akhir pada seluruh data latih)
    set_seed(seed); rng = np.random.RandomState(seed)
    model = build_model(agg, FEAT_DIM[bb]).to(DEVICE)
    opt = make_opt(hp['optimizer'], model.parameters(), hp['lr'], hp['weight_decay'])
    ytr = np.array([Y[w] for w in tr_ids])
    cw = torch.tensor([len(ytr) / (2 * max((ytr == c).sum(), 1)) for c in (0, 1)], dtype=torch.float32, device=DEVICE)
    y_es = np.array([Y[w] for w in es_ids]) if es_ids is not None else None
    n_ep = fixed_epochs if es_ids is None else EPOCHS
    best = dict(auc=-1.0, loss=1e9, epoch=0, state=None); bad = 0; ep = 0
    for ep in range(1, n_ep + 1):
        model.train(); order = rng.permutation(len(tr_ids))
        for s in range(0, len(order), hp['batch_size']):
            chunk = order[s:s + hp['batch_size']]; opt.zero_grad()
            for i in chunk:
                wid = tr_ids[i]; x = BANK[bb][wid]
                if x.shape[0] > MAX_PATCH_TRAIN:
                    x = x[torch.from_numpy(rng.choice(x.shape[0], MAX_PATCH_TRAIN, replace=False))]
                x = x.to(DEVICE).float(); y = torch.tensor([Y[wid]], device=DEVICE)
                logits, aux = model(x, label=Y[wid], instance_eval=(agg == 'CLAM'))
                loss = (F.cross_entropy(logits, y, reduction='none') * cw[y]).sum()
                if agg == 'CLAM':
                    loss = CLAM_BAG_W * loss + (1 - CLAM_BAG_W) * aux['inst_loss']
                (loss / len(chunk)).backward()
            opt.step()
        if es_ids is not None:
            pv = predict(model, es_ids, bb); auc = roc_auc_score(y_es, pv); vl = logloss(y_es, pv)
            if auc > best['auc'] + 1e-4 or (abs(auc - best['auc']) <= 1e-4 and vl < best['loss']):
                best = dict(auc=auc, loss=vl, epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()}); bad = 0
            else:
                bad += 1
            if bad >= PATIENCE: break
    if es_ids is not None:
        model.load_state_dict(best['state']); return model, best['epoch']
    return model, ep

def run_cv(agg, bb, hp, seed):
    # 1) 5-fold CV tingkat pasien → prediksi OOF (160 WSI)   2) threshold dari OOF
    # 3) latih ulang pada seluruh 160 WSI (epoch = median epoch terbaik antar fold)   4) uji pada test (sekali)
    pos = {w: i for i, w in enumerate(ids_pool)}; oof = np.zeros(len(ids_pool)); eps, f_auc, f_thr = [], [], []; t0 = time.time()
    for k, (tr, ho) in enumerate(FOLDS):
        tr_in, es = patient_split(tr, ES_FRACTION, seed + 100 * k)
        model, be = train_one(agg, bb, hp, seed, tr_in, es_ids=es)
        p = predict(model, ho, bb); oof[[pos[w] for w in ho]] = p
        yho = np.array([Y[w] for w in ho]); eps.append(be); f_auc.append(roc_auc_score(yho, p)); f_thr.append(youden_thr(yho, p))
        del model; gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    oof_auc = roc_auc_score(y_pool, oof)
    thr = youden_thr(y_pool, oof) if THRESH_MODE == 'youden_oof' else float(THRESH_MODE)
    n_final = int(max(5, round(float(np.median(eps)))))
    model, _ = train_one(agg, bb, hp, seed, ids_pool, es_ids=None, fixed_epochs=n_final)
    test_probs = predict(model, ids_test, bb); state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return dict(oof_probs=oof, test_probs=test_probs, oof_auc=oof_auc, fold_aucs=f_auc, fold_thr=f_thr, fold_epochs=eps,
                thr=thr, best_epoch=n_final, train_sec=time.time() - t0, state=state)

# ---------- Cache hasil (aman bila runtime terputus) ----------
_fp = hashlib.md5(("|".join(ids_pool + ids_test) + f"v2|{SEED}|{N_FOLDS}|{ES_FRACTION}|{EPOCHS}|{PATIENCE}|{MAX_PATCH_TRAIN}|{THRESH_MODE}|{CLAM_K}|{CLAM_BAG_W}").encode()).hexdigest()
CACHE = f"{RESULT_DIR}/runs_cache_v2.pkl"
RUNS = {}
if os.path.exists(CACHE):
    try:
        _c = pickle.load(open(CACHE, 'rb'))
        if _c.get('fp') == _fp: RUNS = _c['runs']; print(f"♻️  Cache dimuat: {len(RUNS)} run")
        else: print("ℹ️  Cache lama tidak cocok dengan konfigurasi/data sekarang → diabaikan")
    except Exception as e:
        print("ℹ️  Cache tidak terbaca:", e)

def get_run(agg, bb, hp, seed):
    hp = {**DEFAULT_HP, **hp}
    key = (agg, bb, hp['lr'], hp['optimizer'], hp['batch_size'], hp['weight_decay'], seed)
    ck = f"{CKPT_DIR}/{agg}_{bb}_seed{seed}.pt"
    if key not in RUNS or (hp == DEFAULT_HP and not os.path.exists(ck)):
        r = run_cv(agg, bb, hp, seed)
        if hp == DEFAULT_HP: torch.save(r['state'], ck)
        RUNS[key] = {k: v for k, v in r.items() if k != 'state'}
        pickle.dump({'fp': _fp, 'runs': RUNS}, open(CACHE, 'wb'))
    return RUNS[key]

def eval_run(r):
    return cls_metrics(y_test, r['test_probs'], r['thr'])

def seed_runs(agg, bb, hp=None):
    return {s: get_run(agg, bb, hp or {}, s) for s in SEEDS}

def select_seed(runs):                       # dipilih berdasar AUROC OOF-CV (bukan test)
    return max(runs, key=lambda s: runs[s]['oof_auc'])
print(f"✅ Model & prosedur {N_FOLDS}-fold CV siap  (1 run = {N_FOLDS} fold + 1 latih ulang pada {len(ids_pool)} WSI)")

In [ ]:
def summarize(named_runs):
    # -> (df angka per seed, df ringkas: rerata ± SD sampel antar seed)
    long, disp = [], []
    for name, runs in named_runs.items():
        ms = [eval_run(r) for r in runs.values()]
        for s, r, m in zip(runs.keys(), runs.values(), ms):
            long.append(dict(Model=name, Seed=s, OOF_AUROC=r['oof_auc'], AUROC_fold_min=min(r['fold_aucs']), AUROC_fold_max=max(r['fold_aucs']),
                             Threshold_OOF=r['thr'], Thr_fold_min=min(r['fold_thr']), Thr_fold_max=max(r['fold_thr']), Epoch_akhir=r['best_epoch'], **m))
        oo = [r['oof_auc'] for r in runs.values()]
        row = {'Model': name, 'AUROC CV (OOF)': f"{np.mean(oo):.3f} ± {sd(oo):.3f}"}
        for k in METRICS:
            v = [m[k] for m in ms]; row[k] = f"{np.mean(v):.3f} ± {sd(v):.3f}"
        disp.append(row)
    return pd.DataFrame(long), pd.DataFrame(disp)

def rank_check(named_runs):
    # bandingkan urutan model di OOF-CV vs di uji → bahas terbuka bila terbalik
    t = pd.DataFrame({n: dict(OOF_AUROC=np.mean([r['oof_auc'] for r in rr.values()]), Test_AUROC=np.mean([eval_run(r)['AUROC'] for r in rr.values()]))
                      for n, rr in named_runs.items()}).T
    t['Peringkat OOF'] = t.OOF_AUROC.rank(ascending=False).astype(int); t['Peringkat Test'] = t.Test_AUROC.rank(ascending=False).astype(int)
    display(t.round(3))
    if list(t.sort_values('OOF_AUROC', ascending=False).index) != list(t.sort_values('Test_AUROC', ascending=False).index):
        print("⚠️ Urutan model di OOF-CV dan di uji BERBEDA. Laporkan terbuka: model dipilih dari OOF-CV (protokol), "
              "bukan dari uji; selisih kecil dapat berasal dari variasi sampling pada kelas Mutant yang sedikit.")
    else:
        print("✅ Urutan model konsisten antara OOF-CV dan uji.")
    return t

def cm_ax(ax, y, p, thr, color, title):
    cm = confusion_matrix(y, (p >= thr).astype(int), labels=[0, 1])
    pct = cm / cm.sum(1, keepdims=True)
    ann = np.array([[f"{cm[i, j]}\n({pct[i, j]*100:.0f}%)" for j in range(2)] for i in range(2)])
    sns.heatmap(pct, annot=ann, fmt='', cmap=light_cmap(color), vmin=0, vmax=1, cbar=False, ax=ax,
                linewidths=2, linecolor='white', annot_kws={'fontsize': 13, 'fontweight': 'bold', 'color': '#1F2D3D'})
    ax.set_xticklabels(['Wildtype', 'Mutant']); ax.set_yticklabels(['Wildtype', 'Mutant'], rotation=0)
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual'); ax.set_title(title); ax.grid(False)

def compare_figure(specs, title, fname):
    names = [mname(*s) for s in specs]; cols = [color_of(*s) for s in specs]
    runs = {n: seed_runs(*s) for n, s in zip(names, specs)}
    sel  = {n: runs[n][select_seed(runs[n])] for n in names}
    fig = plt.figure(figsize=(20, 10.5)); fig.suptitle(title, fontsize=17, fontweight='bold', y=0.995)
    gs = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.28)
    ax = fig.add_subplot(gs[0, 0])                                                   # (a) ROC
    for n, c in zip(names, cols):
        fpr, tpr, _ = roc_curve(y_test, sel[n]['test_probs'])
        ax.plot(fpr, tpr, color=c, lw=3, label=f"{n} (AUROC {roc_auc_score(y_test, sel[n]['test_probs']):.3f})")
        ax.fill_between(fpr, tpr, alpha=0.08, color=c)
    ax.plot([0, 1], [0, 1], '--', color='#9AA8B8'); ax.set_xlabel('1 − Spesifisitas (FPR)'); ax.set_ylabel('Sensitivitas (TPR)')
    ax.set_title('Kurva ROC — data uji'); ax.legend(loc='lower right')
    ax = fig.add_subplot(gs[0, 1]); w = 0.38; x = np.arange(len(METRICS))            # (b) metrik, rerata ± SD sampel
    for i, (n, c) in enumerate(zip(names, cols)):
        ms = np.array([[eval_run(r)[k] for k in METRICS] for r in runs[n].values()])
        mean = ms.mean(0); sdv = ms.std(0, ddof=1) if len(ms) > 1 else np.zeros(len(METRICS))
        bars = ax.bar(x + (i - 0.5) * w, mean, w, yerr=sdv, capsize=3, color=c, alpha=0.9, label=n, edgecolor='white')
        for b, v in zip(bars, mean):
            ax.text(b.get_x() + b.get_width() / 2, min(v, 1) + 0.03, f"{v:.2f}", ha='center', fontsize=8, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([m.replace('_', '\n') for m in METRICS], fontsize=8.5); ax.set_ylim(0, 1.18)
    ax.set_title(f'Metrik uji (rerata ± SD sampel, {len(SEEDS)} seed)'); ax.legend(loc='lower right', fontsize=8.5)
    ax = fig.add_subplot(gs[0, 2])                                                   # (c) AUROC per seed
    for i, (n, c) in enumerate(zip(names, cols)):
        sv = [r['oof_auc'] for r in runs[n].values()]; st = [eval_run(r)['AUROC'] for r in runs[n].values()]
        ax.scatter([i - 0.12] * len(sv), sv, s=120, facecolor='white', edgecolor=c, lw=2.5, zorder=3, label='CV (OOF)' if i == 0 else None)
        ax.scatter([i + 0.12] * len(st), st, s=120, color=c, edgecolor='white', lw=1.5, zorder=3, label='Test' if i == 0 else None)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names); ax.set_xlim(-0.6, len(names) - 0.4); ax.set_ylabel('AUROC')
    ax.set_title('AUROC per seed (○ CV-OOF, ● test)'); ax.legend(loc='lower right')
    ax = fig.add_subplot(gs[1, 0]); pa, pb = sel[names[0]]['test_probs'], sel[names[1]]['test_probs']   # (d) scatter probabilitas
    ax.scatter(pa[y_test == 0], pb[y_test == 0], s=70, color=C_WT, edgecolor='white', label='Wildtype', alpha=0.9)
    ax.scatter(pa[y_test == 1], pb[y_test == 1], s=70, color=C_MUT, edgecolor='white', label='Mutant', alpha=0.9)
    ax.axvline(sel[names[0]]['thr'], color=cols[0], ls='--'); ax.axhline(sel[names[1]]['thr'], color=cols[1], ls='--')
    ax.set_xlabel(f"P(Mutant) — {names[0]}"); ax.set_ylabel(f"P(Mutant) — {names[1]}"); ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.set_title('Probabilitas per slide (garis putus = threshold OOF)'); ax.legend()
    for j, (n, c) in enumerate(zip(names, cols)):                                    # (e,f) confusion matrix
        cm_ax(fig.add_subplot(gs[1, 1 + j]), y_test, sel[n]['test_probs'], sel[n]['thr'], c, f'Confusion matrix — {n}')
    savefig(fig, fname); plt.show()
    return runs, sel
print("✅ Helper visualisasi siap")

## 🥇 Skenario 1: Model Utama — Swin-T+ABMIL vs Swin-T+CLAM

In [ ]:
SPECS_S1 = [('ABMIL', 'Swin'), ('CLAM', 'Swin')]
runs_s1, sel_s1 = compare_figure(SPECS_S1, 'Skenario 1 — Swin-T+ABMIL vs Swin-T+CLAM', 'S1_ABMIL_vs_CLAM')

long1, disp1 = summarize(runs_s1)
print("\n📋 Hasil data uji (threshold Youden dari prediksi OOF) — rerata ± SD sampel (ddof=1) antar seed")
display(disp1.set_index('Model'))
print("📋 Detail per seed (termasuk rentang AUROC dan threshold antar fold → stabilitas)")
display(long1.round(4))
long1.to_csv(f"{RESULT_DIR}/S1_detail_per_seed.csv", index=False); disp1.to_csv(f"{RESULT_DIR}/S1_ringkasan.csv", index=False)

# Swin-T pilihan CV: dari rerata AUROC OOF-CV (test TIDAK dipakai untuk seleksi)
oof_mean = {a: np.mean([r['oof_auc'] for r in runs_s1[mname(a, 'Swin')].values()]) for a, _ in SPECS_S1}
BEST_AGG = max(oof_mean, key=oof_mean.get)
print(f"\n🏆 Swin-T pilihan CV (AUROC OOF): Swin-T+{BEST_AGG}  | {oof_mean}")
_ = rank_check(runs_s1)

## 🥈 Skenario 2: Perbandingan Swin-T (pilihan CV) vs DenseNet201
Skenario ini adalah **perbandingan**, bukan seleksi. Jika urutan di OOF-CV dan uji berbeda, kode akan memberi peringatan agar dibahas terbuka.

In [ ]:
SPECS_S2 = [(BEST_AGG, 'Swin'), (BEST_AGG, 'Dense')]
runs_s2, sel_s2 = compare_figure(SPECS_S2, f'Skenario 2 — Swin-T+{BEST_AGG} vs DenseNet201+{BEST_AGG}', 'S2_Swin_vs_DenseNet201')

long2, disp2 = summarize(runs_s2)
print("\n📋 Hasil data uji — rerata ± SD sampel (ddof=1) antar seed")
display(disp2.set_index('Model'))
long2.to_csv(f"{RESULT_DIR}/S2_detail_per_seed.csv", index=False); disp2.to_csv(f"{RESULT_DIR}/S2_ringkasan.csv", index=False)
print("\n🔎 Konsistensi urutan OOF-CV vs uji")
_ = rank_check(runs_s2)

NAME_SWIN  = mname(BEST_AGG, 'Swin')
NAME_DENSE = mname(BEST_AGG, 'Dense')

## 🎚️ Skenario 3: Sensitivitas Hyperparameter
Satu faktor diubah per kali (**learning rate**, **optimizer**, **batch size** = jumlah langkah *gradient accumulation*), faktor lain tetap pada `DEFAULT_HP`. Tiap konfigurasi dievaluasi dengan **5-fold CV (AUROC OOF)** dan dilatih ulang pada seluruh data latih untuk dilaporkan pada data uji (seed pertama, 3 model). Konfigurasi default muncul di setiap faktor tetapi hanya **dilatih sekali** (cache). Pemilihan konfigurasi terbaik memakai **OOF-CV**, bukan uji.

In [ ]:
HP_SEED = SEEDS[0]
HP_MODELS = {mname('ABMIL', 'Swin'): ('ABMIL', 'Swin'), mname('CLAM', 'Swin'): ('CLAM', 'Swin'), NAME_DENSE: (BEST_AGG, 'Dense')}
HP_COLORS = {n: color_of(*s) for n, s in HP_MODELS.items()}

jobs = [(m, f, v) for m in HP_MODELS for f, vals in HP_GRID.items() for v in vals]
rows = []
for m, f, v in tqdm(jobs, desc="Skenario 3"):
    agg, bb = HP_MODELS[m]
    r = get_run(agg, bb, {f: v}, HP_SEED); mt = eval_run(r)
    rows.append(dict(Model=m, Faktor=f, Nilai=v, OOF_AUROC=r['oof_auc'], Test_AUROC=mt['AUROC'],
                     Sensitivitas=mt['Sensitivitas'], Spesifisitas=mt['Spesifisitas'], Akurasi=mt['Akurasi'], Best_epoch=r['best_epoch']))
n_unique = 1 + sum(len(v) - 1 for v in HP_GRID.values())
print(f"Konfigurasi unik per model = {n_unique}; total = {n_unique * len(HP_MODELS)} run unik (masing-masing {N_FOLDS} fold + 1 latih ulang)")
df_hp = pd.DataFrame(rows)
df_hp.to_csv(f"{RESULT_DIR}/S3_hyperparameter.csv", index=False)
print("✅ Selesai")

In [ ]:
# ---------- Kurva performa vs konfigurasi ----------
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6)); fig.suptitle('Skenario 3 — Sensitivitas Hyperparameter (AUROC)', fontsize=16, fontweight='bold', y=1.02)
titles = {'lr': 'Learning rate', 'optimizer': 'Optimizer', 'batch_size': 'Batch size (bag/update)'}
for ax, (f, vals) in zip(axes, HP_GRID.items()):
    sub = df_hp[df_hp.Faktor == f]
    if f == 'optimizer':
        w = 0.26; x = np.arange(len(vals))
        for i, m in enumerate(HP_MODELS):
            s = sub[sub.Model == m].set_index('Nilai').loc[vals]
            b = ax.bar(x + (i - 1) * w, s.Test_AUROC, w, color=HP_COLORS[m], label=m, edgecolor='white')
            ax.scatter(x + (i - 1) * w, s.OOF_AUROC, marker='D', s=40, color='white', edgecolor=HP_COLORS[m], lw=1.8, zorder=3)
            for xi, v in zip(x + (i - 1) * w, s.Test_AUROC): ax.text(xi, v + 0.01, f"{v:.2f}", ha='center', fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels(vals)
    else:
        for m in HP_MODELS:
            s = sub[sub.Model == m].set_index('Nilai').loc[vals]; xs = np.arange(len(vals))
            ax.plot(xs, s.Test_AUROC, '-o', color=HP_COLORS[m], lw=3, ms=9, label=f"{m} (test)")
            ax.plot(xs, s.OOF_AUROC, '--s', color=HP_COLORS[m], lw=1.6, ms=6, alpha=0.6, label=f"{m} (OOF)")
        ax.set_xticks(xs); ax.set_xticklabels([f"{v:g}" for v in vals])
        # tandai default
        dv = DEFAULT_HP[f]; ax.axvline(vals.index(dv), color='#FFC83D', lw=10, alpha=0.25, zorder=0)
    ax.set_ylim(0.4, 1.05); ax.set_ylabel('AUROC'); ax.set_xlabel(titles[f]); ax.set_title(titles[f])
axes[0].legend(fontsize=8, ncol=1, loc='lower right'); axes[1].legend(fontsize=8, loc='lower right')
fig.text(0.5, -0.04, "Sorotan kuning = nilai default · bar = AUROC test · ◇ = AUROC CV (OOF)", ha='center', fontsize=10, color='#566573')
plt.tight_layout(); savefig(fig, 'S3_hyperparameter_curves'); plt.show()

# ---------- Heatmap AUROC / Sensitivitas / Spesifisitas ----------
df_hp['Konfigurasi'] = df_hp.Faktor.map({'lr': 'LR', 'optimizer': 'Opt', 'batch_size': 'BS'}) + " = " + df_hp.Nilai.map(lambda v: f"{v:g}" if not isinstance(v, str) else v)
order = list(dict.fromkeys(df_hp.Konfigurasi))
fig, axes = plt.subplots(1, 3, figsize=(19, 7))
for ax, (col, ttl, cm) in zip(axes, [('Test_AUROC', 'AUROC (test)', 'YlGnBu'), ('Sensitivitas', 'Sensitivitas (test)', 'YlOrRd'), ('Spesifisitas', 'Spesifisitas (test)', 'PuBuGn')]):
    pv = df_hp.pivot_table(index='Konfigurasi', columns='Model', values=col).loc[order, list(HP_MODELS)]
    sns.heatmap(pv, annot=True, fmt='.3f', cmap=cm, vmin=0.3, vmax=1, linewidths=1.5, linecolor='white', ax=ax, cbar_kws={'shrink': 0.7})
    ax.set_title(ttl); ax.set_ylabel(''); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=15)
plt.tight_layout(); savefig(fig, 'S3_hyperparameter_heatmap'); plt.show()

print("\n📋 Tabel performa vs konfigurasi")
display(df_hp.drop(columns='Konfigurasi').round(4).style.background_gradient(subset=['OOF_AUROC', 'Test_AUROC', 'Sensitivitas', 'Spesifisitas', 'Akurasi'], cmap='YlGnBu', vmin=0.4, vmax=1))
for m in HP_MODELS:
    s = df_hp[df_hp.Model == m].sort_values('OOF_AUROC', ascending=False, kind='stable').iloc[0]
    print(f"🔎 {m}: konfigurasi terbaik menurut CV (OOF) → {s.Faktor} = {s.Nilai} (OOF AUROC {s.OOF_AUROC:.3f}, Test AUROC {s.Test_AUROC:.3f})")

## ⚡ Skenario 4: Efisiensi Komputasi
Pipeline: **Swin-T+ABMIL**, **Swin-T+CLAM**, dan **DenseNet201+agregator pilihan CV**. Definisi pengukuran (eksplisit):
- **Parameter**: total (backbone + agregator) dan **trainable**. Backbone dibekukan (hanya dipakai untuk ekstraksi fitur) sehingga *parameter trainable = agregator MIL*.
- **FLOPs/patch**: `torch.utils.flop_counter` (FLOPs = 2×MAC). Backbone: 1 patch 224×224. Agregator: bag sintetis 1000 patch dibagi 1000.
- **Waktu inferensi MIL** (fitur → probabilitas): diukur **per slide uji nyata** (setiap slide diulang 10×, setelah pemanasan), dilaporkan **rerata ± SD antar slide**. Ini tahap yang dibandingkan antar agregator.
- **Estimasi waktu ekstraksi** = waktu/patch (rerata ± SD dari 10 ulangan batch) × median jumlah patch. Ini **estimasi**, bukan pengukuran end-to-end: tidak termasuk pembacaan WSI, tiling, dan normalisasi Macenko.
- **Puncak memori GPU** diukur **terpisah** untuk dua tahap: (a) ekstraksi satu batch di backbone saja, (b) forward MIL pada slide uji terbesar (agregator saja). Klaim hemat memori antar agregator hanya berlaku untuk tahap (b).
- Seluruh pengukuran memakai **satu perangkat yang sama**; nama GPU dicetak dan dicatat di tabel. Samakan dengan perangkat yang dilaporkan pada ekstraksi fitur.

In [ ]:
import timm
import torchvision.models as tv_models
from torch.utils.flop_counter import FlopCounterMode

class DenseNet201Features(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = tv_models.densenet201(weights=None).features
    def forward(self, x):
        x = F.relu(self.features(x), inplace=False)
        return torch.flatten(F.adaptive_avg_pool2d(x, 1), 1)

def build_backbone(bb):
    return timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=0) if bb == 'Swin' else DenseNet201Features()

n_params = lambda m: sum(p.numel() for p in m.parameters())
def sync():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
def count_flops(fn):
    try:
        with FlopCounterMode(display=False) as fc: fn()
        return fc.get_total_flops()
    except Exception as e:
        print("⚠️ FLOPs gagal dihitung:", e); return float('nan')
def peak_mem(fn):
    if DEVICE.type != 'cuda': return float('nan')
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); fn(); sync()
    return torch.cuda.max_memory_allocated() / 2**20
def free_gpu():
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'
print(f"🖥️  Perangkat pengukuran: {GPU_NAME}  (catat di laporan; harus sama dengan perangkat yang dilaporkan)")
BENCH_BS = 64 if DEVICE.type == 'cuda' else 8
N_WARM, N_REP = 3, 10
N_MED = int(df_test.n_patches.median()); big_id = df_test.sort_values('n_patches').wsi_id.iloc[-1]

PIPES = [('ABMIL', 'Swin'), ('CLAM', 'Swin'), (BEST_AGG, 'Dense')]
rows4 = []
for agg, bb in tqdm(PIPES, desc="Skenario 4"):
    bbm = build_backbone(bb).eval(); agm = build_model(agg, FEAT_DIM[bb]).eval()
    p_bb, p_ag = n_params(bbm), n_params(agm)
    with torch.no_grad():
        fl_bb = count_flops(lambda: bbm(torch.randn(1, 3, 224, 224)))
        fl_ag = count_flops(lambda: agm(torch.randn(1000, FEAT_DIM[bb]))) / 1000
    # ---- (a) ekstraksi: hanya backbone di GPU ----
    bbm = bbm.to(DEVICE); xb = torch.randn(BENCH_BS, 3, 224, 224, device=DEVICE); ts = []
    with torch.no_grad():
        for _ in range(N_WARM): bbm(xb)
        sync()
        for _ in range(N_REP):
            t = time.perf_counter(); bbm(xb); sync(); ts.append((time.perf_counter() - t) / BENCH_BS)
        mem_ext = peak_mem(lambda: bbm(xb))
    t_patch, t_patch_sd = float(np.mean(ts)), sd(ts)
    del xb; bbm = bbm.cpu(); free_gpu()
    # ---- (b) MIL: fitur → probabilitas, hanya agregator di GPU ----
    agm = agm.to(DEVICE); per_slide = []
    with torch.no_grad():
        for w in ids_test:
            x = BANK[bb][w].to(DEVICE).float()
            for _ in range(2): F.softmax(agm(x)[0], 1)
            sync(); tt = []
            for _ in range(N_REP):
                t = time.perf_counter(); F.softmax(agm(x)[0], 1); sync(); tt.append(time.perf_counter() - t)
            per_slide.append(float(np.mean(tt)))
        xbig = BANK[bb][big_id].to(DEVICE).float()
        mem_mil = peak_mem(lambda: agm(xbig))
    del xbig, agm, bbm; free_gpu()
    rows4.append({'Model': mname(agg, bb), 'Backbone': bb, 'Agg': agg,
                  'Param total (M)': (p_bb + p_ag) / 1e6, 'Param backbone beku (M)': p_bb / 1e6, 'Param trainable (M)': p_ag / 1e6,
                  'GFLOPs/patch backbone': fl_bb / 1e9, 'GFLOPs/patch agregator': fl_ag / 1e9, 'GFLOPs/patch total': (fl_bb + fl_ag) / 1e9,
                  'MIL ms/slide (rerata)': np.mean(per_slide) * 1e3, 'MIL ms/slide (SD)': sd(per_slide) * 1e3,
                  'Ekstraksi ms/patch (rerata)': t_patch * 1e3, 'Ekstraksi ms/patch (SD)': t_patch_sd * 1e3,
                  'Estimasi ekstraksi s/slide': t_patch * N_MED, 'Estimasi ekstraksi s/slide (SD)': t_patch_sd * N_MED,
                  'Memori puncak ekstraksi (MB)': mem_ext, 'Memori puncak MIL (MB)': mem_mil, 'Perangkat': GPU_NAME})

df4 = pd.DataFrame(rows4); df4.to_csv(f"{RESULT_DIR}/S4_efisiensi.csv", index=False)
print(f"Median patch/slide uji: {N_MED} | slide terbesar: {int(df_test.n_patches.max())} patch | slide uji untuk benchmark MIL: {len(ids_test)}")
disp4 = pd.DataFrame({'Model': df4.Model,
    'Param total / trainable (M)': [f"{a:.2f} / {b:.2f}" for a, b in zip(df4['Param total (M)'], df4['Param trainable (M)'])],
    'GFLOPs/patch (backbone + agregator)': [f"{a:.3f} + {b:.4f}" for a, b in zip(df4['GFLOPs/patch backbone'], df4['GFLOPs/patch agregator'])],
    'MIL (ms/slide)': [f"{a:.2f} ± {b:.2f}" for a, b in zip(df4['MIL ms/slide (rerata)'], df4['MIL ms/slide (SD)'])],
    'Estimasi ekstraksi (s/slide)': [f"{a:.2f} ± {b:.2f}" for a, b in zip(df4['Estimasi ekstraksi s/slide'], df4['Estimasi ekstraksi s/slide (SD)'])],
    'Memori puncak ekstraksi / MIL (MB)': [f"{a:,.0f} / {b:,.0f}" for a, b in zip(df4['Memori puncak ekstraksi (MB)'], df4['Memori puncak MIL (MB)'])]})
display(disp4.set_index('Model').T)

In [ ]:
cols4 = [color_of(a, b) for a, b in PIPES]; names4 = df4.Model.tolist(); xx = np.arange(len(names4))
fig, axes = plt.subplots(1, 5, figsize=(25, 5.4)); fig.suptitle(f'Skenario 4 — Efisiensi Komputasi ({GPU_NAME})', fontsize=16, fontweight='bold', y=1.04)

def stacked(ax, c_bb, c_ag, ttl, unit, fmt):
    ax.bar(xx, df4[c_bb], color='#FFE29A', edgecolor='white', label='Backbone (beku)')
    ax.bar(xx, df4[c_ag], bottom=df4[c_bb], color=cols4, edgecolor='white', label='Agregator MIL (trainable)')
    for i, (a, b) in enumerate(zip(df4[c_bb], df4[c_ag])): ax.text(i, a + b, fmt.format(a + b), ha='center', va='bottom', fontweight='bold')
    ax.set_xticks(xx); ax.set_xticklabels(names4, rotation=12); ax.set_title(ttl); ax.set_ylabel(unit); ax.legend(fontsize=8)
    ax.set_ylim(0, (df4[c_bb] + df4[c_ag]).max() * 1.2)
stacked(axes[0], 'Param backbone beku (M)', 'Param trainable (M)', 'Jumlah parameter', 'Juta parameter', '{:.2f}')
stacked(axes[1], 'GFLOPs/patch backbone', 'GFLOPs/patch agregator', 'FLOPs per patch', 'GFLOPs / patch', '{:.2f}')

ax = axes[2]; ax.bar(xx, df4['MIL ms/slide (rerata)'], yerr=df4['MIL ms/slide (SD)'], capsize=6, color=cols4, edgecolor='white')
for i, (m, s_) in enumerate(zip(df4['MIL ms/slide (rerata)'], df4['MIL ms/slide (SD)'])): ax.text(i, m + s_, f"{m:.1f} ± {s_:.1f}", ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.set_xticks(xx); ax.set_xticklabels(names4, rotation=12); ax.set_title('Waktu inferensi MIL\n(fitur → probabilitas, per slide uji)'); ax.set_ylabel('ms / slide')
ax.set_ylim(0, (df4['MIL ms/slide (rerata)'] + df4['MIL ms/slide (SD)']).max() * 1.25)

ax = axes[3]; ax.bar(xx, df4['Estimasi ekstraksi s/slide'], yerr=df4['Estimasi ekstraksi s/slide (SD)'], capsize=6, color=cols4, edgecolor='white', hatch='//', alpha=0.85)
for i, (m, s_) in enumerate(zip(df4['Estimasi ekstraksi s/slide'], df4['Estimasi ekstraksi s/slide (SD)'])): ax.text(i, m + s_, f"{m:.2f} ± {s_:.2f}", ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.set_xticks(xx); ax.set_xticklabels(names4, rotation=12); ax.set_title(f'ESTIMASI waktu ekstraksi fitur\n(ms/patch × {N_MED} patch; tanpa I/O WSI & Macenko)'); ax.set_ylabel('detik / slide')
ax.set_ylim(0, (df4['Estimasi ekstraksi s/slide'] + df4['Estimasi ekstraksi s/slide (SD)']).max() * 1.25)

ax = axes[4]; w = 0.38
ax.bar(xx - w / 2, df4['Memori puncak ekstraksi (MB)'], w, color=cols4, edgecolor='white')
ax.bar(xx + w / 2, df4['Memori puncak MIL (MB)'], w, color=cols4, edgecolor='white', alpha=0.55, hatch='..')
for i in range(len(xx)):
    for dx, c in [(-w / 2, 'Memori puncak ekstraksi (MB)'), (w / 2, 'Memori puncak MIL (MB)')]:
        v = df4[c].iloc[i]; ax.text(i + dx, 0 if np.isnan(v) else v, 'n/a' if np.isnan(v) else f"{v:,.0f}", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.set_xticks(xx); ax.set_xticklabels(names4, rotation=12); ax.set_title('Puncak memori GPU per tahap'); ax.set_ylabel('MB')
ax.legend(handles=[Patch(fc='#888', label='Ekstraksi (backbone, 1 batch)'), Patch(fc='#888', alpha=0.55, hatch='..', label='MIL (slide terbesar)')], fontsize=8)
plt.tight_layout(); savefig(fig, 'S4_efisiensi'); plt.show()

## 📐 Skenario 5: Uji Signifikansi Statistik Agregat
Perbandingan berpasangan pada **data uji yang sama**:
1. Swin-T+ABMIL vs Swin-T+CLAM
2. Swin-T pilihan CV vs DenseNet201

Uji: **DeLong** (ΔAUROC), **McNemar** (hasil klasifikasi berpasangan pada threshold OOF; *exact binomial* bila diskordan < 25, selain itu χ² dengan koreksi kontinuitas), dan **bootstrap** (resampling **per pasien**, ≥1000×) untuk 95% CI semua metrik serta CI selisih AUROC. Signifikan bila **p < 0,05**.

**Koreksi multiple-testing:** Bonferroni (α/2) diterapkan **per jenis uji** terhadap 2 perbandingan. DeLong (AUROC) dan McNemar (keputusan klasifikasi pada threshold) menjawab pertanyaan berbeda sehingga masing-masing diperlakukan sebagai satu keluarga; p-value bootstrap hanya konfirmasi atas CI selisih AUROC (bukan uji ketiga yang berdiri sendiri).

In [ ]:
# ---------- DeLong (Sun & Xu, 2014) ----------
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    T2 = np.empty(N); T2[J] = T; return T2

def _fast_delong(preds_sorted, m):
    n = preds_sorted.shape[1] - m; k = preds_sorted.shape[0]
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    tx, ty, tz = (np.empty((k, m)), np.empty((k, n)), np.empty((k, m + n)))
    for r in range(k):
        tx[r], ty[r], tz[r] = _midrank(pos[r]), _midrank(neg[r]), _midrank(preds_sorted[r])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    return aucs, cov

def delong_test(y, p1, p2):
    order = np.argsort(-y, kind='stable'); m = int(y.sum())
    aucs, cov = _fast_delong(np.vstack([p1, p2])[:, order], m)
    diff = aucs[0] - aucs[1]; var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 1e-15: return dict(auc_a=aucs[0], auc_b=aucs[1], diff=diff, z=0.0, p=1.0 if abs(diff) < 1e-12 else 0.0)
    z = diff / np.sqrt(var)
    return dict(auc_a=aucs[0], auc_b=aucs[1], diff=diff, z=z, p=2 * norm.sf(abs(z)))

def mcnemar_test(y, pa, pb, ta, tb):
    ca = ((pa >= ta).astype(int) == y); cb = ((pb >= tb).astype(int) == y)
    n11, b, c, n00 = int((ca & cb).sum()), int((ca & ~cb).sum()), int((~ca & cb).sum()), int((~ca & ~cb).sum())
    if b + c == 0: p, stat, method = 1.0, 0.0, 'n/a (tanpa diskordan)'
    elif b + c < 25: p, stat, method = binomtest(min(b, c), b + c, 0.5).pvalue, float(min(b, c)), 'exact binomial'
    else:
        stat = (abs(b - c) - 1) ** 2 / (b + c); p, method = chi2.sf(stat, 1), 'χ² (koreksi kontinuitas)'
    return dict(n11=n11, b=b, c=c, n00=n00, stat=stat, p=p, method=method)

# ---------- Sanity check implementasi ----------
_r = np.random.RandomState(0); _y = np.r_[np.ones(40), np.zeros(60)].astype(int)
_p1 = np.r_[_r.normal(1.0, 1, 40), _r.normal(0, 1, 60)]
assert abs(_fast_delong(np.vstack([_p1, _p1])[:, np.argsort(-_y, kind='stable')], 40)[0][0] - roc_auc_score(_y, _p1)) < 1e-9
print("✅ DeLong cocok dengan roc_auc_score")

In [ ]:
# ---------- Model terpilih per pipeline (seed dengan AUROC validation terbaik) ----------
S5 = {}
for agg, bb in [('ABMIL', 'Swin'), ('CLAM', 'Swin'), (BEST_AGG, 'Dense')]:
    n = mname(agg, bb); rr = seed_runs(agg, bb); s = select_seed(rr)
    S5[n] = dict(run=rr[s], seed=s, color=color_of(agg, bb), p=rr[s]['test_probs'], thr=rr[s]['thr'])
names5 = list(S5)
print("Model & seed terpilih:", {n: S5[n]['seed'] for n in names5})
print(f"Slide uji: {len(y_test)} | Pasien uji: {df_test.patient_id.nunique()} | Mutant: {int(y_test.sum())} | Wildtype: {int((1 - y_test).sum())}")
if df_test.patient_id.nunique() < len(y_test):
    print("ℹ️ Ada pasien dengan >1 slide → bootstrap memakai resampling per PASIEN (cluster); DeLong/McNemar mengasumsikan slide independen.")

# ---------- Bootstrap (cluster per pasien, indeks sama untuk semua model → berpasangan) ----------
rng = np.random.default_rng(SEED)
pid = df_test.patient_id.values; upat = np.unique(pid); grp = {u: np.where(pid == u)[0] for u in upat}
BOOT = []
while len(BOOT) < N_BOOT:
    idx = np.concatenate([grp[upat[i]] for i in rng.integers(0, len(upat), len(upat))])
    if y_test[idx].min() != y_test[idx].max(): BOOT.append(idx)
FGRID = np.linspace(0, 1, 101)
bm = {n: {k: np.empty(N_BOOT) for k in METRICS} for n in names5}; broc = {n: np.empty((N_BOOT, len(FGRID))) for n in names5}
for b, idx in enumerate(tqdm(BOOT, desc=f"Bootstrap ×{N_BOOT}")):
    for n in names5:
        m = cls_metrics(y_test[idx], S5[n]['p'][idx], S5[n]['thr'])
        for k in METRICS: bm[n][k][b] = m[k]
        f, t, _ = roc_curve(y_test[idx], S5[n]['p'][idx]); broc[n][b] = np.interp(FGRID, f, t)

lo, hi = 100 * ALPHA / 2, 100 * (1 - ALPHA / 2)
rows = []
for n in names5:
    pt = cls_metrics(y_test, S5[n]['p'], S5[n]['thr']); r = {'Model': n}
    for k in METRICS:
        a, c = np.nanpercentile(bm[n][k], [lo, hi]); r[k] = f"{pt[k]:.3f} [{a:.3f}–{c:.3f}]"; r[k + '_pt'], r[k + '_lo'], r[k + '_hi'] = pt[k], a, c
    rows.append(r)
df_ci = pd.DataFrame(rows)
print(f"\n📋 Metrik data uji dengan {100*(1-ALPHA):.0f}% CI bootstrap (B={N_BOOT})")
display(df_ci[['Model'] + METRICS].set_index('Model'))
df_ci.to_csv(f"{RESULT_DIR}/S5_bootstrap_CI.csv", index=False)

In [ ]:
# ---------- Uji berpasangan ----------
COMPARISONS = [(mname('ABMIL', 'Swin'), mname('CLAM', 'Swin')), (NAME_SWIN, NAME_DENSE)]
K_COMP = len(COMPARISONS); res_rows, MC = [], {}
for a, b in COMPARISONS:
    dl = delong_test(y_test, S5[a]['p'], S5[b]['p'])
    mc = mcnemar_test(y_test, S5[a]['p'], S5[b]['p'], S5[a]['thr'], S5[b]['thr']); MC[(a, b)] = mc
    d = bm[a]['AUROC'] - bm[b]['AUROC']; dlo, dhi = np.nanpercentile(d, [lo, hi])
    p_boot = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    res_rows.append({'Perbandingan': f"{a}  vs  {b}",
                     'AUROC A': dl['auc_a'], 'AUROC B': dl['auc_b'], 'ΔAUROC': dl['diff'],
                     f'ΔAUROC {100*(1-ALPHA):.0f}% CI (bootstrap)': f"[{dlo:.3f}, {dhi:.3f}]",
                     'DeLong z': dl['z'], 'p DeLong': dl['p'], 'p DeLong (Bonferroni)': min(1.0, dl['p'] * K_COMP),
                     'Signifikan DeLong (Bonferroni)': 'Ya' if dl['p'] * K_COMP < ALPHA else 'Tidak',
                     'McNemar b (A✓B✗)': mc['b'], 'McNemar c (A✗B✓)': mc['c'], 'Metode McNemar': mc['method'],
                     'p McNemar': mc['p'], 'p McNemar (Bonferroni)': min(1.0, mc['p'] * K_COMP),
                     'Signifikan McNemar (Bonferroni)': 'Ya' if mc['p'] * K_COMP < ALPHA else 'Tidak',
                     'p bootstrap (ΔAUROC)': p_boot})
df_tests = pd.DataFrame(res_rows); df_tests.to_csv(f"{RESULT_DIR}/S5_uji_signifikansi.csv", index=False)
print(f"📋 Uji signifikansi (α = {ALPHA}; Bonferroni α/{K_COMP} = {ALPHA/K_COMP:.3f} per jenis uji, {K_COMP} perbandingan)")
print("   Alasan: DeLong (AUROC) dan McNemar (keputusan klasifikasi) = dua keluarga uji berbeda; bootstrap p hanya konfirmasi CI.")
display(df_tests.set_index('Perbandingan').T)

In [ ]:
# ---------- Visualisasi Skenario 5 (A): CI semua metrik utama, ROC band, ΔAUROC ----------
fig = plt.figure(figsize=(23, 10.8)); fig.suptitle('Skenario 5 — Signifikansi Statistik & Bootstrap 95% CI', fontsize=17, fontweight='bold', y=0.995)
gs = fig.add_gridspec(2, 4, hspace=0.45, wspace=0.32)
for j, k in enumerate(['AUROC', 'AUPRC', 'Sensitivitas', 'Spesifisitas']):
    ax = fig.add_subplot(gs[0, j]); ypos = np.arange(len(names5))[::-1]
    for yy, n in zip(ypos, names5):
        r = df_ci[df_ci.Model == n].iloc[0]
        ax.errorbar(r[k + '_pt'], yy, xerr=[[r[k + '_pt'] - r[k + '_lo']], [r[k + '_hi'] - r[k + '_pt']]], fmt='o', ms=13,
                    color=S5[n]['color'], ecolor=S5[n]['color'], elinewidth=4, capsize=7, capthick=2.5, mec='white', mew=2)
        ax.text(r[k + '_pt'], yy + 0.22, f"{r[k + '_pt']:.3f}\n[{r[k + '_lo']:.2f}–{r[k + '_hi']:.2f}]", ha='center', va='bottom', fontsize=8.5)
    if k == 'AUPRC':
        ax.axvline(y_test.mean(), color='#E63946', ls=':', lw=2); ax.text(y_test.mean(), -0.55, f"baseline = {y_test.mean():.2f}", color='#E63946', fontsize=8.5, ha='center')
    ax.set_yticks(ypos); ax.set_yticklabels(names5 if j == 0 else [''] * len(names5)); ax.set_ylim(-0.7, len(names5) - 0.2); ax.set_xlim(0.0, 1.1)
    ax.set_title(f'{k} (95% CI)'); ax.set_xlabel(k)
ax = fig.add_subplot(gs[1, 0:2])
for n in names5:
    c = S5[n]['color']; l, h = np.percentile(broc[n], [lo, hi], axis=0); _f, _t, _ = roc_curve(y_test, S5[n]['p']); mean_t = np.interp(FGRID, _f, _t)
    ax.plot(FGRID, mean_t, color=c, lw=3, label=f"{n} ({df_ci[df_ci.Model == n].AUROC_pt.iloc[0]:.3f})"); ax.fill_between(FGRID, l, h, color=c, alpha=0.18)
ax.plot([0, 1], [0, 1], '--', color='#9AA8B8'); ax.set_xlabel('1 − Spesifisitas'); ax.set_ylabel('Sensitivitas'); ax.set_title('ROC + pita CI 95% bootstrap'); ax.legend(loc='lower right', fontsize=9)
for j, (a, b) in enumerate(COMPARISONS):
    ax = fig.add_subplot(gs[1, 2 + j]); d = bm[a]['AUROC'] - bm[b]['AUROC']; dlo, dhi = np.nanpercentile(d, [lo, hi]); r = df_tests.iloc[j]
    ax.hist(d, bins=40, color=S5[a]['color'], alpha=0.75, edgecolor='white'); ax.axvspan(dlo, dhi, color='#FFD166', alpha=0.3, label=f'CI 95% [{dlo:.3f}, {dhi:.3f}]')
    ax.axvline(0, color='#E63946', lw=2.5, ls='--', label='Δ = 0'); ax.axvline(d.mean(), color='#1F2D3D', lw=2, label=f'rerata Δ = {d.mean():.3f}')
    ax.set_title(f'ΔAUROC bootstrap\n{a} − {b}', fontsize=10.5); ax.set_xlabel('ΔAUROC'); ax.set_ylabel('Frekuensi'); ax.legend(fontsize=8)
    ax.text(0.02, 0.95, f"DeLong p = {r['p DeLong']:.4f}", transform=ax.transAxes, va='top', fontsize=9.5, fontweight='bold', color='#1F2D3D')
savefig(fig, 'S5a_bootstrap_CI_ROC_delta'); plt.show()

# ---------- Visualisasi Skenario 5 (B): p-value & McNemar ----------
fig, axes = plt.subplots(1, 3, figsize=(19, 5.4)); fig.suptitle('Skenario 5 — Ringkasan p-value & Tabel McNemar', fontsize=16, fontweight='bold', y=1.04)
ax = axes[0]; tests = [('DeLong', 'p DeLong'), ('McNemar', 'p McNemar'), ('Bootstrap Δ', 'p bootstrap (ΔAUROC)')]; w = 0.36
for j, ((a, b), cc) in enumerate(zip(COMPARISONS, [C_ABMIL, C_DENSE])):
    for i, (_, col) in enumerate(tests):
        v = df_tests.iloc[j][col]; h = -np.log10(max(v, 1e-6)); ax.bar(i + (j - 0.5) * w, h, w, color=cc, edgecolor='white', label=f"{a} vs {b}" if i == 0 else None)
        ax.text(i + (j - 0.5) * w, h + 0.03, f"p={v:.3f}", ha='center', fontsize=8.5, fontweight='bold')
ax.axhline(-np.log10(ALPHA), color='#E63946', ls='--', lw=2); ax.axhline(-np.log10(ALPHA / K_COMP), color='#FF9F1C', ls='--', lw=2)
ax.text(2.45, -np.log10(ALPHA) + 0.03, f'α = {ALPHA}', color='#E63946', ha='right', fontweight='bold'); ax.text(2.45, -np.log10(ALPHA / K_COMP) + 0.03, f'Bonferroni α/{K_COMP}', color='#FF9F1C', ha='right', fontweight='bold')
ax.set_xticks(range(3)); ax.set_xticklabels([t for t, _ in tests]); ax.set_ylabel('−log10(p)'); ax.set_title('Di atas garis = signifikan'); ax.legend(fontsize=8, loc='upper left')
for ax, (a, b) in zip(axes[1:], COMPARISONS):
    mc = MC[(a, b)]; M = np.array([[mc['n11'], mc['b']], [mc['c'], mc['n00']]])
    sns.heatmap(M, annot=True, fmt='d', cmap=light_cmap(S5[a]['color']), cbar=False, linewidths=3, linecolor='white', ax=ax, annot_kws={'fontsize': 20, 'fontweight': 'bold', 'color': '#1F2D3D'})
    ax.set_xticklabels([f'{b}\nbenar', f'{b}\nsalah']); ax.set_yticklabels([f'{a}\nbenar', f'{a}\nsalah'], rotation=0)
    ax.set_title(f"McNemar: p = {mc['p']:.4f} ({mc['method']})", fontsize=10.5); ax.grid(False)
plt.tight_layout(); savefig(fig, 'S5b_pvalue_mcnemar'); plt.show()

## 🔬 Analisis Kualitatif (3.6.2): Distribusi Atensi & Error Analysis
Memakai model Swin-T pilihan CV (seed terpilih). Setiap slide uji dikelompokkan **TP / TN / FP / FN** pada threshold OOF. Untuk tiap slide dihitung **entropi atensi ternormalisasi** (0 = terpusat pada sedikit patch, 1 = merata) dan **massa atensi pada 5% patch teratas**. Heatmap spasial dibuat bila H5 menyimpan koordinat patch; jika tidak, hanya statistik atensi yang ditampilkan.

In [ ]:
agg_q, bb_q = BEST_AGG, 'Swin'; seed_q = S5[NAME_SWIN]['seed']
ck = f"{CKPT_DIR}/{agg_q}_{bb_q}_seed{seed_q}.pt"
qual = None
if not os.path.exists(ck):
    print("⚠️ Checkpoint tidak ditemukan:", ck)
else:
    model = build_model(agg_q, FEAT_DIM[bb_q]).to(DEVICE); model.load_state_dict(torch.load(ck, map_location=DEVICE)); model.eval()
    pt, thr = S5[NAME_SWIN]['p'], S5[NAME_SWIN]['thr']; pred = (pt >= thr).astype(int)
    outcome = np.where((y_test == 1) & (pred == 1), 'TP', np.where((y_test == 0) & (pred == 0), 'TN', np.where((y_test == 0) & (pred == 1), 'FP', 'FN')))
    recs, ATT = [], {}
    with torch.no_grad():
        for i, w in enumerate(ids_test):
            _, aux = model(BANK[bb_q][w].to(DEVICE).float()); a = aux['attention'][0].cpu().numpy().astype(np.float64); ATT[w] = a; n = len(a)
            ent = float(-(a * np.log(a + 1e-12)).sum() / np.log(n)) if n > 1 else 0.0
            top = float(np.sort(a)[::-1][:max(1, int(0.05 * n))].sum())
            recs.append(dict(wsi_id=w, label='Mutant' if y_test[i] else 'Wildtype', P_mutant=pt[i], prediksi='Mutant' if pred[i] else 'Wildtype',
                             hasil=outcome[i], n_patch=n, entropi_atensi=ent, massa_atensi_top5pct=top))
    qual = pd.DataFrame(recs); qual.to_csv(f"{RESULT_DIR}/S6_analisis_kualitatif.csv", index=False)
    print(f"Model: {NAME_SWIN} (seed {seed_q}) | threshold OOF = {thr:.3f}")
    print("\n❌ Slide salah klasifikasi (FP / FN):")
    display(qual[qual.hasil.isin(['FP', 'FN'])].sort_values('hasil').round(3))
    print("📋 Ringkasan atensi per kelompok hasil")
    display(qual.groupby('hasil')[['P_mutant', 'n_patch', 'entropi_atensi', 'massa_atensi_top5pct']].agg(['mean', 'std', 'count']).round(3))

    PAL = {'TP': '#22B573', 'TN': '#2D9CDB', 'FP': '#FF9F1C', 'FN': '#FF4D6D'}; order = [o for o in ['TP', 'TN', 'FP', 'FN'] if o in set(qual.hasil)]
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.4)); fig.suptitle('Analisis Kualitatif — Perilaku Atensi per Hasil Klasifikasi', fontsize=16, fontweight='bold', y=1.03)
    for ax, col, ttl in [(axes[0], 'entropi_atensi', 'Entropi atensi (0 = terpusat, 1 = merata)'), (axes[1], 'massa_atensi_top5pct', 'Massa atensi pada 5% patch teratas')]:
        sns.boxplot(data=qual, x='hasil', y=col, order=order, palette=PAL, ax=ax, width=0.5, fliersize=0, linewidth=1.5)
        sns.stripplot(data=qual, x='hasil', y=col, order=order, color='#1F2D3D', size=6, alpha=0.7, ax=ax)
        ax.set_title(ttl); ax.set_xlabel('Hasil klasifikasi'); ax.set_ylabel('')
    ax = axes[2]
    for o in order:
        curves = []
        for w in qual[qual.hasil == o].wsi_id:
            a = np.sort(ATT[w])[::-1]; xs = np.linspace(0, 100, 200); curves.append(np.interp(xs, np.linspace(0, 100, len(a)), np.cumsum(a)))
        m = np.mean(curves, 0); ax.plot(xs, m, color=PAL[o], lw=3, label=f"{o} (n={len(curves)})")
    ax.set_xlabel('% patch teratas (urut atensi)'); ax.set_ylabel('Atensi kumulatif'); ax.set_title('Kurva atensi kumulatif (rerata)'); ax.legend()
    plt.tight_layout(); savefig(fig, 'S6_atensi_kualitatif'); plt.show()

    # ---- heatmap spasial bila ada koordinat ----
    def get_coords(wid):
        with h5py.File(f"{FEAT_SWIN}/{wid}.h5", 'r') as hf:
            for k in ('coords', 'coordinates', 'coord', 'xy', 'positions'):
                if k in hf: return hf[k][:]
            return None
    cand = []
    for o in ['TP', 'FN', 'FP', 'TN']:
        sub = qual[qual.hasil == o]
        if len(sub): cand.append((o, sub.sort_values('P_mutant', ascending=(o in ('FN', 'TN'))).wsi_id.iloc[0]))
    shown = [(o, w, get_coords(w)) for o, w in cand]
    if all(c is None for _, _, c in shown):
        with h5py.File(f"{FEAT_SWIN}/{ids_test[0]}.h5", 'r') as hf: print("ℹ️ H5 tidak berisi koordinat patch; kunci yang ada:", list(hf.keys()), "→ heatmap spasial dilewati.")
    else:
        shown = [t for t in shown if t[2] is not None]
        fig, axes = plt.subplots(1, len(shown), figsize=(5.2 * len(shown), 5)); axes = np.atleast_1d(axes)
        for ax, (o, w, xy) in zip(axes, shown):
            a = ATT[w]; sc = ax.scatter(xy[:, 0], -xy[:, 1], c=a / a.max(), s=14, cmap='YlOrRd', marker='s')
            ax.set_title(f"{o} — {w[:16]}\nP(Mutant) = {qual.set_index('wsi_id').loc[w, 'P_mutant']:.2f}", fontsize=10.5); ax.set_aspect('equal'); ax.axis('off')
        fig.colorbar(sc, ax=axes.tolist(), shrink=0.7, label='atensi relatif'); savefig(fig, 'S6_heatmap_atensi'); plt.show()

## 💾 Sel Akhir: Ekspor Semua Tabel ke Excel

In [ ]:
xlsx = f"{RESULT_DIR}/Hasil_5_Skenario.xlsx"
try:
    with pd.ExcelWriter(xlsx) as xw:
        long1.to_excel(xw, sheet_name='S1_per_seed', index=False);  disp1.to_excel(xw, sheet_name='S1_ringkasan', index=False)
        long2.to_excel(xw, sheet_name='S2_per_seed', index=False);  disp2.to_excel(xw, sheet_name='S2_ringkasan', index=False)
        fold_tab.to_excel(xw, sheet_name='CV_komposisi_fold', index=False)
        df_hp.drop(columns='Konfigurasi').to_excel(xw, sheet_name='S3_hyperparameter', index=False)
        df4.to_excel(xw, sheet_name='S4_efisiensi', index=False)
        df_ci[['Model'] + METRICS].to_excel(xw, sheet_name='S5_bootstrap_CI', index=False)
        df_tests.to_excel(xw, sheet_name='S5_uji_signifikansi', index=False)
        if 'qual' in globals() and qual is not None: qual.to_excel(xw, sheet_name='S6_kualitatif', index=False)
    print("✅ Excel:", xlsx)
except Exception as e:
    print("⚠️ Excel gagal:", e, "(CSV tetap tersimpan)")
print("\n📁 Isi", RESULT_DIR)
for p in sorted(Path(RESULT_DIR).rglob('*')):
    if p.is_file() and p.suffix in ('.csv', '.xlsx', '.png'): print("  ", p.relative_to(RESULT_DIR))